# Analisis Karakteristik Waktu–Frekuensi Komponen Instrumen pada Remake Soundtrack Bad Piggies Menggunakan FFT dan STFT dengan Variasi Panjang Jendela

Dataset utama adalah lima kelompok instrumen **hasil rekonstruksi MIDI + GeneralUser GS**, dengan not dan penempatan Playlist dari FLP. Ini bukan render FLEX/DirectWave asli. DRUMS dan VOCALS turut dianalisis melalui WAV hasil dekode MP3 pada Bagian 9. Konversi itu tidak meningkatkan kualitas sumber. Tidak ada not acak atau pemisahan instrumen otomatis.

Jalankan notebook ini dari atas ke bawah. Seluruh kode persiapan perangkat, ekstraksi FLP, rekonstruksi MIDI/WAV, konversi DRUMS/VOCALS, FFT, STFT, validasi, serta ekspor gambar dan tabel terdapat langsung dalam sel notebook. Penjelasan sumber, metode, hasil, dan keterbatasan disatukan dalam paper ETS AFW.pdf. Petunjuk menjalankan komputasi terdapat pada Bagian 0 notebook ini.

## 0. Lingkungan Komputasi

Buka dengan kernel Python 3.11 atau lingkungan yang menyediakan dependensi pada requirements.txt. Folder proyek dicari dari lokasi kerja kernel, termasuk ketika salinan notebook dibuka dari folder proyek.

### Susunan Berkas dan Eksekusi

Pilih kernel Python, lalu **Restart Kernel and Run All**. Gunakan sel `%pip` di bawah hanya jika dependensi belum tersedia. Seluruh kode ada di notebook ini; hasil perhitungan ditulis ke folder berikut.

| Folder | Isi |
| --- | --- |
| `data/sumber/` | FLP, MP3, dan sampel asli |
| `data/soundfont/` | SoundFont, lisensi, dan asal unduhan |
| `gambar/instrumen/`, `gambar/kanal_audio/` | Seluruh grafik menurut kelompok analisis |
| `hasil/instrumen/`, `hasil/kanal_audio/` | Tabel CSV/LaTeX dan ringkasan numerik |
| `hasil/rekonstruksi/`, `hasil/midi/` | Ekstraksi not, audit Playlist, validasi, dan MIDI |
| `hasil/audio/` | WAV instrumen, kanal sumber, dan cuplikan |
| `perangkat/` | Pustaka Python lokal dan FluidSynth untuk menjalankan notebook |
| `laporan/` | Bagian-bagian sumber LaTeX paper |

`REBUILD_AUDIO=False` memakai WAV yang sudah tersedia; jika WAV belum ada, notebook membuatnya. `RECALCULATE=True` dan `RECALCULATE_SOURCE=True` menghitung ulang analisis. Sesudah perangkat tersedia, perhitungan tidak memerlukan jaringan. PDF dikompilasi dari ETS AFW.tex dengan pdflatex dua kali.

Sebelum menyimpan notebook untuk Git, gunakan **Clear All Outputs**. Gambar dan tabel tetap tersedia dalam folder hasilnya; tidak perlu menyimpan notebook keluaran duplikat. Aturan pengecualian berada pada `.gitignore` root repository.


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True

candidates = [Path.cwd(), *Path.cwd().parents]
candidates += [p / 'Semester 9/Analisis Fourier dan Wavelet/ETS Project' for p in candidates.copy()]
ROOT = next((p.resolve() for p in candidates
             if (p / 'ETS AFW.tex').exists() and (p / 'data/sumber_data.json').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Buka notebook dari ETS Project atau root workspace Tugas Kuliah.')
local_packages = str(ROOT / 'perangkat/python')
if local_packages not in sys.path:
    sys.path.append(local_packages)
print('Folder proyek:', ROOT)

### Dependensi Python

Jika paket belum terpasang, hilangkan tanda # pada baris %pip di bawah dan jalankan selnya. Setelah instalasi, restart kernel lalu jalankan semua sel. Pada workspace ini dependensi sudah tersedia. Sel tersebut tetap dikomentari agar pengulangan analisis tidak memerlukan jaringan.

In [ ]:
# %pip install -r "{ROOT / 'requirements.txt'}"

In [ ]:
from collections import Counter, defaultdict
import ctypes
import hashlib
import io
import json
import math
import os
import platform
import struct
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import scipy
from scipy import fft, signal
import soundfile as sf
import pyflp
import mido
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from IPython.display import display, Image, Audio

## 1. Sumber Data dan Validasi Dataset

FLP yang diperiksa mempunyai 156 BPM, PPQ 96, tujuh pattern dan 12 klip Playlist. Posisi not harus memperhitungkan posisi klip **dan offset pemotongan**. Rumusnya `tick_global = posisi_klip + tick_not_pattern - offset_sumber`. CSV menyimpan nilai asli agar dapat diaudit.

Aktifkan `REBUILD_AUDIO` bila ingin mengulangi ekstraksi, penulisan MIDI, dan rendering. Dalam keadaan normal, audio yang sudah divalidasi dipakai kembali. Integritas WAV dibandingkan dengan hash hasil validasi.

### Perangkat Sintesis dan SoundFont

FluidSynth 2.6.1 portabel untuk Windows dan GeneralUser GS disimpan lokal. Fungsi berikut mengunduhnya dari sumber resmi hanya jika berkas yang diperlukan belum tersedia. Commit SoundFont, URL, ukuran, dan hash dicatat dalam data/soundfont/sumber.json. Koneksi internet hanya diperlukan saat pengadaan pertama.

In [ ]:
SOUNDFONT_COMMIT = "684543d5e5efaef08d02be50dcda8d552478fa60"
FLUIDSYNTH_URL = "https://github.com/FluidSynth/fluidsynth/releases/download/v2.6.1/fluidsynth-v2.6.1-win10-x64-cpp11.zip"
TOOL_SOURCES = {
    "data/soundfont/GeneralUser-GS.sf2": f"https://raw.githubusercontent.com/mrbumpy409/GeneralUser-GS/{SOUNDFONT_COMMIT}/GeneralUser-GS.sf2",
    "data/soundfont/LICENSE.txt": f"https://raw.githubusercontent.com/mrbumpy409/GeneralUser-GS/{SOUNDFONT_COMMIT}/documentation/LICENSE.txt",
}

def prepare_tools():
    manifest = {}
    for relative, url in TOOL_SOURCES.items():
        target = ROOT / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        if not target.exists():
            request = urllib.request.Request(url, headers={"User-Agent": "AFW-research"})
            with urllib.request.urlopen(request, timeout=60) as response:
                target.write_bytes(response.read())
        manifest[relative] = {"url": url, "sha256": hashlib.sha256(target.read_bytes()).hexdigest(), "bytes": target.stat().st_size}
    destination = (ROOT / "perangkat/fluidsynth").resolve()
    if not next(destination.rglob("libfluidsynth-3.dll"), None):
        request = urllib.request.Request(FLUIDSYNTH_URL, headers={"User-Agent": "AFW-research"})
        with urllib.request.urlopen(request, timeout=60) as response:
            archive_bytes = response.read()
        destination.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
            for item in archive.infolist():
                target = (destination / item.filename).resolve()
                if not target.is_relative_to(destination):
                    raise ValueError("Unsafe archive path")
            archive.extractall(destination)
        manifest["perangkat/fluidsynth"] = {"url": FLUIDSYNTH_URL, "archive_sha256": hashlib.sha256(archive_bytes).hexdigest()}
    else:
        manifest["perangkat/fluidsynth"] = {"url": FLUIDSYNTH_URL, "version": "2.6.1"}
    (ROOT / "data/soundfont/sumber.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("FluidSynth dan SoundFont siap.")

### Parameter Rekonstruksi dan Pembacaan Sumber

Kelompok kanal dan program General MIDI ditetapkan secara eksplisit. Pembaca event dan sampel berikut khusus mendukung paket FLP ini. Sampel Classical Acoustic didekode di memori untuk audit, tanpa membuat salinan WAV pendek yang tidak digunakan dalam analisis.

In [ ]:
RECON_OUTPUT = ROOT / "hasil/rekonstruksi"
RECON_FS = 44100
RECON_GROUPS = {0: "Brass", 2: "Guitar", 3: "Piano", 4: "Piano", 5: "Piano",
          6: "Guitar", 7: "Guitar", 9: "Bass", 10: "Tuba"}
GM_PROGRAMS = {"Piano": (0, "Acoustic Grand Piano"), "Guitar": (24, "Nylon Guitar"),
            "Bass": (32, "Acoustic Bass"), "Brass": (61, "Brass Section"), "Tuba": (58, "Tuba")}
INSTRUMENT_ORDER = ["Bass", "Piano", "Guitar", "Brass", "Tuba"]

def write_json(path, obj):
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")

def raw_events(path):
    data = path.read_bytes()
    assert data[:4] == b"FLhd"
    head_size = struct.unpack_from("<I", data, 4)[0]
    p = 8 + head_size
    assert data[p:p+4] == b"FLdt"
    size = struct.unpack_from("<I", data, p+4)[0]
    p += 8
    end = p+size
    assert end == len(data)
    while p < end:
        eid = data[p]; p += 1
        if eid < 192:
            n = (1, 2, 4)[eid//64]
        else:
            n = shift = 0
            while True:
                v = data[p]; p += 1
                n |= (v & 127) << shift; shift += 7
                if v < 128: break
        assert p+n <= end
        yield eid, data[p:p+n]
        p += n

def decode_samples(folder):
    rows = []
    for file in sorted(folder.glob("Classical Acoustic*.wav")):
        data = file.read_bytes(); p = 12; chunks = {}
        assert data[:4] == b"RIFF" and data[8:12] == b"WAVE"
        while p+8 <= len(data):
            kind, n = struct.unpack_from("<4sI", data, p)
            chunks[kind] = data[p+8:p+8+n]; p += 8+n+(n%2)
        tag = struct.unpack_from("<H", chunks[b"fmt "])[0]
        assert tag == 0x674f and chunks[b"data"].startswith(b"OggS")
        audio, fs = sf.read(io.BytesIO(chunks[b"data"]), always_2d=True)
        rows.append({"file": file.name, "format_tag": "0x674f", "codec": "Ogg Vorbis in RIFF/WAVE",
                     "fs": fs, "samples": len(audio), "duration_seconds": len(audio)/fs,
                     "peak": float(abs(audio).max()), "role": "sampel pendek; tidak dipakai sebagai track penuh"})
    pd.DataFrame(rows).to_csv(RECON_OUTPUT / "audit_sampel.csv", index=False)
    return rows

### Ekstraksi Not dan Penempatan Playlist

Fungsi ini mempertahankan pitch, velocity, dan posisi not yang berasal dari FLP. Batas klip, offset sumber, dan indeks pattern diperiksa sebelum CSV timeline ditulis.

In [ ]:
def extract():
    RECON_OUTPUT.mkdir(parents=True, exist_ok=True)
    folder = ROOT / "data/sumber"
    source = next(folder.glob("*.flp"))
    project = pyflp.parse(source)
    assert project.version.major == 25, "Parser Playlist ini ditujukan untuk berkas FL25 yang diperiksa."
    events = list(raw_events(source))
    playlists = [d for eid,d in events if eid == 233]
    assert len(playlists) == 1 and len(playlists[0]) == 960
    # Konversi waktu tidak menebak automation: dataset ini mempunyai satu event tempo.
    tempos = [struct.unpack("<I", d)[0]/1000 for eid,d in events if eid == 156]
    assert tempos == [156.0]
    tempo = mido.bpm2tempo(project.tempo)
    ppq = project.ppq
    assert ppq == 96 and project.main_pitch == 0
    channels = {c.iid:c for c in project.channels}
    patterns = {p.iid:p for p in project.patterns}
    channel_rows = []
    for c in channels.values():
        row = {"channel_id": c.iid, "name": c.name, "plugin": getattr(c,"internal_name",None),
               "type": type(c).__name__, "enabled": c.enabled, "group": RECON_GROUPS.get(c.iid, "tidak dirender")}
        if c.iid < 11:
            row.update(root=c.keyboard.root_note, pitch_cents=c.pitch_shift, volume_fl=c.volume, pan_fl=c.pan)
            assert c.keyboard.root_note == 60 and c.pitch_shift == 0
        channel_rows.append(row)
    pd.DataFrame(channel_rows).to_csv(RECON_OUTPUT/"channels.csv", index=False)
    tracks = {}
    for eid,d in events:
        if eid == 238:  # TrackID.Data
            tracks[struct.unpack_from("<I", d)[0]] = bool(d[12])
    clips = []
    for index in range(len(playlists[0])//80):
        d = playlists[0][index*80:(index+1)*80]
        pos, base, item, length, reverse, group = struct.unpack_from("<IHHIHH",d)
        flags = struct.unpack_from("<H",d,18)[0]
        assert base == 20480 and flags == 64 and 0 <= reverse < 500
        assert tracks.get(500-reverse, True), "Track nonaktif membutuhkan penanganan tersendiri."
        is_pattern = item > base
        start, end = struct.unpack_from("<ii",d,24) if is_pattern else struct.unpack_from("<ff",d,24)
        clip = {"clip_id": index, "track": 500-reverse, "kind": "pattern" if is_pattern else "channel",
                "item_id": item-base if is_pattern else item, "position_ticks": pos,
                "length_ticks": length, "offset_start_raw": start, "offset_end_raw": end, "flags": flags}
        if is_pattern:
            assert item-base in patterns
            clip["name"] = patterns[item-base].name
            clip["source_start_ticks"] = max(0,start)
            assert end == -1 or end-start == length
        else:
            assert item in channels
            clip["name"] = channels[item].name
        clips.append(clip)
    pd.DataFrame(clips).to_csv(RECON_OUTPUT/"playlist.csv", index=False)
    write_json(RECON_OUTPUT/"playlist.json", clips)
    raw_notes = []
    for pattern in patterns.values():
        assert not pattern.looped and not list(pattern.controllers)
        for ni,note in enumerate(pattern.notes):
            n = dict(note._item)
            assert n["fine_pitch"] == 120 and int(n["flags"]) == 16384
            n.update(pattern_id=pattern.iid, pattern_name=pattern.name, pattern_note_id=ni)
            n["flags"] = int(n["flags"])
            raw_notes.append(n)
    pd.DataFrame(raw_notes).drop(columns=["_io"],errors="ignore").to_csv(RECON_OUTPUT/"not_pattern.csv",index=False)
    midi_channels = {cid:i for i,cid in enumerate(sorted(RECON_GROUPS))}
    placed = []
    clipped_count = 0
    for clip in clips:
        if clip["kind"] != "pattern": continue
        begin = clip["source_start_ticks"]
        end = begin+clip["length_ticks"]
        for n in raw_notes:
            if n["pattern_id"] != clip["item_id"]: continue
            a = n["position"]; b = a+n["length"]
            if a >= end or b <= begin: continue
            assert a >= begin, "Not sudah berbunyi sebelum batas klip; perlu kebijakan retrigger eksplisit."
            cid = n["rack_channel"]
            assert cid in RECON_GROUPS and channels[cid].enabled
            start = clip["position_ticks"]+a-begin
            finish = clip["position_ticks"]+min(b,end)-begin
            clipped_count += b > end
            velocity = int(np.clip(np.floor(n["velocity"]*127/128+.5),1,127))
            placed.append({"group":RECON_GROUPS[cid], "channel_id":cid,"channel_name":channels[cid].name,
                "midi_channel":midi_channels[cid],"pattern_id":n["pattern_id"],"clip_id":clip["clip_id"],
                "pattern_note_id":n["pattern_note_id"],"pitch_midi":n["key"],"velocity_fl":n["velocity"],
                "velocity_midi":velocity,"start_tick":start,"end_tick":finish,"duration_tick":finish-start,
                "start_seconds":mido.tick2second(start,ppq,tempo),
                "end_seconds":mido.tick2second(finish,ppq,tempo)})
    placed.sort(key=lambda n:(n["start_tick"],n["channel_id"],n["pitch_midi"]))
    assert len(placed)==len(raw_notes)==2219
    pd.DataFrame(placed).to_csv(RECON_OUTPUT/"not_timeline.csv",index=False)
    last = max(c["position_ticks"]+c["length_ticks"] for c in clips)
    timeline = mido.tick2second(last,ppq,tempo)
    metadata = {"source_flp":str(source.relative_to(ROOT)),"sha256":hashlib.sha256(source.read_bytes()).hexdigest(),
        "fl_version":str(project.version),"tempo_bpm":project.tempo,"tempo_midi_us":tempo,"ppq":ppq,
        "time_signature":"4/4","playlist_record_bytes":80,"playlist_records":len(clips),
        "pattern_clips":sum(c["kind"]=="pattern" for c in clips),"patterns":len(patterns),
        "pattern_repetitions":dict(Counter(c["name"] for c in clips if c["kind"]=="pattern")),
        "notes":len(placed),"clipped_notes":clipped_count,"timeline_ticks":last,
        "timeline_seconds_exact_bpm":last/ppq*60/project.tempo,"timeline_seconds_midi":timeline,
        "midi_tempo_quantization_seconds":abs(timeline-last/ppq*60/project.tempo),
        "sample_rate":RECON_FS,"release_tail_seconds":2.0,"programs_zero_based":GM_PROGRAMS,
        "tempo_evidence":"Satu event tempo 156 BPM; tidak ada controller pattern; automation bernama volume saja.",
        "limitations":["Preset SoundFont pengganti, bukan FLEX/DirectWave asli.",
            "Volume/pan/mixer/efek/automation asli tidak direplikasi; CC7=100, pan=64, gain=0.2 seragam.",
            "Velocity FL 0..128 dikonversi proporsional ke MIDI 0..127; nilai mentah disimpan.",
            "MP3 penuh tidak digabung; klip Playlist dan berkas penuh memiliki durasi berbeda."]}
    metadata["samples"] = decode_samples(folder)
    write_json(RECON_OUTPUT/"metadata.json",metadata)
    return placed,metadata

### Penulisan MIDI dan Validasi Not

MIDI dibaca kembali setelah ditulis. Pitch, velocity, kanal, serta tick mulai dan akhir setiap not harus cocok dengan hasil ekstraksi.

In [ ]:
def write_midi(notes,meta,name):
    midi = mido.MidiFile(type=1,ticks_per_beat=meta["ppq"])
    conductor = mido.MidiTrack();midi.tracks.append(conductor)
    conductor.extend([mido.MetaMessage("track_name",name="Tempo"),mido.MetaMessage("set_tempo",tempo=meta["tempo_midi_us"]),
                      mido.MetaMessage("time_signature",numerator=4,denominator=4),
                      mido.MetaMessage("end_of_track",time=meta["timeline_ticks"])])
    for cid in sorted({n["channel_id"] for n in notes}):
        subset=[n for n in notes if n["channel_id"]==cid];channel=subset[0]["midi_channel"]
        tr=mido.MidiTrack();midi.tracks.append(tr)
        tr.extend([mido.MetaMessage("track_name",name=subset[0]["channel_name"]),
                   mido.Message("program_change",channel=channel,program=GM_PROGRAMS[subset[0]["group"]][0]),
                   mido.Message("control_change",channel=channel,control=7,value=100),
                   mido.Message("control_change",channel=channel,control=10,value=64)])
        events=[]
        for n in subset:
            events.extend([(n["start_tick"],1,n["pitch_midi"],n["velocity_midi"]),
                           (n["end_tick"],0,n["pitch_midi"],0)])
        previous=0
        for tick,on,key,vel in sorted(events):
            tr.append(mido.Message("note_on" if on else "note_off",channel=channel,note=key,velocity=vel,time=tick-previous));previous=tick
        tr.append(mido.MetaMessage("end_of_track",time=meta["timeline_ticks"]-previous))
    path=ROOT/"hasil/midi"/f"{name}.mid";path.parent.mkdir(parents=True, exist_ok=True);midi.save(path)
    # Verifikasi round-trip MIDI: bukan hanya jumlah not, juga pitch, velocity, dan kedua tick.
    recovered=[]
    for track in mido.MidiFile(path).tracks:
        tick=0;active={}
        for msg in track:
            tick+=msg.time
            if msg.type=="note_on" and msg.velocity:
                assert (msg.channel,msg.note) not in active,"Not se-pitch bertumpuk di kanal sama."
                active[msg.channel,msg.note]=(tick,msg.velocity)
            elif msg.type=="note_off" or msg.type=="note_on":
                a,v=active.pop((msg.channel,msg.note));recovered.append((msg.channel,msg.note,v,a,tick))
        assert not active
    expected=[(n["midi_channel"],n["pitch_midi"],n["velocity_midi"],n["start_tick"],n["end_tick"]) for n in notes]
    assert sorted(recovered)==sorted(expected)
    return path

### Sintesis WAV dan Campuran Instrumen

FluidSynth merender partitur melalui preset pengganti, dengan chorus dan reverb dimatikan. Semua instrumen memakai 44.100 Hz dan timeline yang sama; campuran hanya menjumlahkan lima WAV. Pemeriksaan meliputi jumlah not, waktu mulai, amplitudo, dan clipping.

In [ ]:
def render_midi(path,meta):
    binary=next((ROOT/"perangkat/fluidsynth").rglob("libfluidsynth-3.dll"),None)
    handle=None
    if binary:
        os.environ["PATH"]=str(binary.parent)+os.pathsep+os.environ["PATH"]
        handle=os.add_dll_directory(str(binary.parent))
    import fluidsynth
    synth=fluidsynth.Synth(samplerate=RECON_FS,gain=.2)
    synth.setting("synth.reverb.active",0);synth.setting("synth.chorus.active",0)
    sfid=synth.sfload(str(ROOT/"data/soundfont/GeneralUser-GS.sf2"))
    assert sfid>=0
    write_float=fluidsynth._fl.fluid_synth_write_float
    write_float.argtypes=[ctypes.c_void_p,ctypes.c_int,ctypes.c_void_p,ctypes.c_int,ctypes.c_int,ctypes.c_void_p,ctypes.c_int,ctypes.c_int]
    write_float.restype=ctypes.c_int
    total=round((meta["timeline_seconds_midi"]+meta["release_tail_seconds"])*RECON_FS)
    audio=np.zeros((total,2),dtype=np.float32)
    def fill(a,b):
        if b<=a:return
        block=np.empty((b-a,2),dtype=np.float32)
        address=block.ctypes.data
        assert write_float(synth.synth,b-a,address,0,2,address,1,2)==0
        audio[a:b]=block
    tick=0;previous=0;messages=0
    for msg in mido.merge_tracks(mido.MidiFile(path).tracks):
        tick+=msg.time
        sample=round(mido.tick2second(tick,meta["ppq"],meta["tempo_midi_us"])*RECON_FS)
        fill(previous,sample);previous=sample
        if msg.type=="program_change":synth.program_select(msg.channel,sfid,0,msg.program)
        elif msg.type=="control_change":synth.cc(msg.channel,msg.control,msg.value)
        elif msg.type=="note_on" and msg.velocity:synth.noteon(msg.channel,msg.note,msg.velocity);messages+=1
        elif msg.type in ("note_on","note_off"):synth.noteoff(msg.channel,msg.note)
    fill(previous,total);synth.delete()
    if handle:handle.close()
    return audio,messages

def reconstruct_audio():
    notes,meta=extract()
    wavdir=ROOT/"hasil/audio/instrumen";wavdir.mkdir(parents=True, exist_ok=True)
    validation=[];audio_sum=None
    for group in INSTRUMENT_ORDER:
        part=[n for n in notes if n["group"]==group]
        path=write_midi(part,meta,group)
        audio,count=render_midi(path,meta)
        assert count==len(part) and np.isfinite(audio).all()
        peak=float(abs(audio).max());rms=float(np.sqrt(np.mean(audio.astype(float)**2)))
        assert peak>1e-5 and rms>1e-6 and peak<1
        first=min(n["start_seconds"] for n in part)
        pre=audio[:max(0,int(first*RECON_FS)-64)]
        assert not pre.size or abs(pre).max()<1e-6
        nonzero=np.flatnonzero(abs(audio).max(axis=1)>1e-5)
        onset=nonzero[0]/RECON_FS
        assert onset>=first-1/RECON_FS and onset-first<.1
        sf.write(wavdir/f"{group}.wav",audio,RECON_FS,subtype="FLOAT")
        mono=audio.mean(axis=1,dtype=np.float64)
        if audio_sum is None:audio_sum=audio.astype(np.float64)
        else:audio_sum+=audio
        row={"instrument":group,"notes":count,"duration_seconds":len(audio)/RECON_FS,"fs":RECON_FS,
             "first_note_seconds":first,"audio_onset_threshold_seconds":onset,
             "last_note_end_seconds":max(n["end_seconds"] for n in part),
             "onset_delay_ms":(onset-first)*1000,"pitch_min":min(n["pitch_midi"] for n in part),
             "pitch_max":max(n["pitch_midi"] for n in part),"peak_stereo":peak,"rms_mono":float(np.sqrt(np.mean(mono**2))),
             "midi_roundtrip_exact":True,"silent":False,"clipped":False,
             "sha256_wav":hashlib.sha256((wavdir/f"{group}.wav").read_bytes()).hexdigest()}
        validation.append(row);print(group,count,"notes",round(first,6),"start",round(peak,4),"peak",flush=True)
    write_midi(notes,meta,"All_Instruments")
    gain=min(1.0,.98/float(abs(audio_sum).max()))
    sf.write(wavdir/"All_Instruments.wav",audio_sum*gain,RECON_FS,subtype="FLOAT")
    meta.update(render_engine="FluidSynth 2.6.1",soundfont="GeneralUser GS 2.0.3",mix_gain=gain,
                mix_contains=INSTRUMENT_ORDER,midi_roundtrip_all_passed=True)
    write_json(RECON_OUTPUT/"metadata.json",meta)
    pd.DataFrame(validation).to_csv(RECON_OUTPUT/"validasi_audio.csv",index=False)
    write_json(RECON_OUTPUT/"validasi_audio.json",validation)
    print("Selesai; MP3 tidak dimasukkan ke All_Instruments.wav.")

### Pelaksanaan Rekonstruksi dan Pemeriksaan Dataset

REBUILD_AUDIO=False memakai WAV yang tersedia dan memeriksa hash-nya. WAV yang belum tersedia otomatis dibuat; ubah menjadi True untuk mengekstrak ulang FLP dan merender seluruh instrumen.

In [ ]:
REBUILD_AUDIO = False
required_audio = [ROOT/'hasil/rekonstruksi/metadata.json'] + [ROOT/'hasil/audio/instrumen'/f'{name}.wav' for name in ['Bass','Piano','Guitar','Brass','Tuba','All_Instruments']]
if REBUILD_AUDIO or not all(path.exists() for path in required_audio):
    binary = next((ROOT/'perangkat/fluidsynth').rglob('libfluidsynth-3.dll'), None)
    if binary is None or not (ROOT/'data/soundfont/GeneralUser-GS.sf2').exists():
        prepare_tools()
    reconstruct_audio()
meta = json.loads((ROOT/'hasil/rekonstruksi/metadata.json').read_text(encoding='utf-8'))
validasi = pd.read_csv(ROOT/'hasil/rekonstruksi/validasi_audio.csv')
for row in validasi.itertuples():
    wav = ROOT/'hasil/audio/instrumen'/f'{row.instrument}.wav'
    assert hashlib.sha256(wav.read_bytes()).hexdigest() == row.sha256_wav
assert validasi.midi_roundtrip_exact.all() and not validasi.silent.any() and not validasi.clipped.any()
display(validasi[['instrument','notes','first_note_seconds','last_note_end_seconds','duration_seconds','fs','onset_delay_ms']])
display(pd.read_csv(ROOT/'hasil/rekonstruksi/playlist.csv'))
print('Timeline MIDI:', meta['timeline_seconds_midi'], 's; total not:', meta['notes'])


## 2. Pelaksanaan Analisis

Semua WAV memakai 44.100 Hz dan panjang yang sama. Stereo dirata-ratakan menjadi mono, lalu rerata globalnya dikurangi untuk analisis spektral. Keaktifan diperiksa **sebelum** pengurangan rerata agar bagian yang semula nol tidak menjadi aktif palsu karena offset. Tidak ada normalisasi puncak per instrumen.

Sel berikut menjalankan FFT, dua STFT, pemeriksaan energi dan ekspor grafik/tabel. `RECALCULATE=True` adalah pilihan untuk reproduksi penuh; `False` hanya membaca hasil tersimpan.

### Parameter Analisis dan Fungsi Spektral

Fungsi berikut menghitung FFT satu sisi dengan bobot DC/Nyquist yang sesuai, kerapatan daya, konversi dB, dan penjumlahan pita frekuensi. Fungsi yang sama digunakan untuk lima instrumen dan dua kanal audio sumber.

In [ ]:
INSTRUMENT_OUTPUT=ROOT/"hasil/instrumen"
INSTRUMENT_FIGURES=ROOT/"gambar/instrumen"
CLIPS=ROOT/"hasil/audio/cuplikan"
INSTRUMENT_NAMES=["Bass","Piano","Guitar","Brass","Tuba"]
INSTRUMENT_COLORS=["#246b91","#b75332","#358363","#805da0","#ac811c"]
BANDS=[(0,250),(250,1000),(1000,4000),(4000,22050)]
ZOOM=(29.0,33.0)
INSTRUMENT_DB_LIMITS=(-110,-35)
plt.rcParams.update({"font.size":10,"axes.spines.top":False,"axes.spines.right":False,
                     "figure.dpi":120,"savefig.dpi":170,"font.family":"DejaVu Sans"})

def prepare_analysis_folders():
    for p in (INSTRUMENT_OUTPUT,INSTRUMENT_FIGURES,CLIPS):p.mkdir(parents=True, exist_ok=True)

def save_figure(fig,name,directory=INSTRUMENT_FIGURES):
    fig.savefig(directory/name,bbox_inches="tight",facecolor="white");plt.close(fig)

def one_sided_weights(n):
    c=np.full(n//2+1,2.0);c[0]=1
    if n%2==0:c[-1]=1
    return c

def power_db(x):return 10*np.log10(np.maximum(x,1e-18))

def frequency_band_mask(nu,lo,hi):return (nu>=lo)&((nu<hi) if hi<22050 else (nu<=hi))

def spectrum(x,fs):
    F=fft.rfft(x);nu=fft.rfftfreq(len(x),1/fs)
    P=one_sided_weights(len(x))*abs(F)**2/(fs*len(x))
    return nu,P,F

def spectral_envelope(nu,P,step=1):
    labels=np.floor(nu/step).astype(int)
    idx=np.r_[0,np.flatnonzero(np.diff(labels))+1]
    return nu[idx],np.maximum.reduceat(P,idx)

def write_instrument_table(name,headers,rows,align=None):
    align=align or "l"+"r"*(len(headers)-1)
    text="\\begin{tabular}{"+align+"}\n\\toprule\n"+" & ".join(headers)+" \\\\\n\\midrule\n"
    text+="\n".join(" & ".join(map(str,row))+" \\\\" for row in rows)
    text+="\n\\bottomrule\n\\end{tabular}\n"
    (INSTRUMENT_OUTPUT/name).write_text(text,encoding="utf-8")

### Analisis Satu Berkas Audio

Urutan perhitungan meliputi mono dan pengurangan rerata, RMS, FFT global, identitas Parseval, segmentasi waktu, serta STFT Hann 1024/4096. Grafik dan cuplikan dibuat dari WAV yang sama; ambang aktivitas diperiksa pada mono sebelum reratanya dikurangi.

In [ ]:
def analyse_one(name,color,source=None,output=INSTRUMENT_OUTPUT,figure_dir=INSTRUMENT_FIGURES,clip_dir=CLIPS,
                frequency_max=2000,db_limits=INSTRUMENT_DB_LIMITS,fft_limits=(-135,-25)):
    source=Path(source) if source is not None else ROOT/"hasil/audio/instrumen"/f"{name}.wav"
    a,fs=sf.read(source,always_2d=True)
    assert fs==44100 and a.shape[1]==2 and np.isfinite(a).all()
    mono=a.mean(axis=1);mu=float(mono.mean());x=mono-mu;n=len(x)
    # Cuplikan mempertahankan amplitudo rekonstruksi, tanpa normalisasi per instrumen.
    sf.write(clip_dir/f"{name}_29-33s.wav",a[round(ZOOM[0]*fs):round(ZOOM[1]*fs)],fs,subtype="PCM_16")
    del a
    q=round(.1*fs);starts=np.arange(0,n,q);counts=np.minimum(q,n-starts)
    rms=np.sqrt(np.add.reduceat(x*x,starts)/counts);tr=(starts+counts/2)/fs
    # Keaktifan diperiksa sebelum pengurangan mean agar ekor nol tidak menjadi
    # aktif palsu akibat offset -mean, khususnya pada SoundFont Tuba.
    rms_raw=np.sqrt(np.add.reduceat(mono*mono,starts)/counts)
    active=rms_raw>=rms_raw.max()*.01
    active_rms=np.sqrt(np.sum(rms[active]**2*counts[active])/counts[active].sum())
    nu,P,F=spectrum(x,fs);df=fs/n
    energy=float(np.sum(x*x)/fs);ef=float(np.sum(one_sided_weights(n)*abs(F)**2)/(fs*n))
    error=abs(energy-ef)/energy
    assert error<1e-11 and np.isclose(np.sum(P)*df,np.mean(x*x),rtol=1e-11)
    peaks,_=signal.find_peaks(P,distance=int(np.ceil(10/df)))
    peaks=peaks[(nu[peaks]>=20)&(nu[peaks]<=10000)]
    chosen=peaks[np.argsort(P[peaks])[-6:][::-1]]
    peakrows=[{"instrument":name,"rank":i+1,"frequency_hz":float(nu[k]),"omega_rad_s":float(2*np.pi*nu[k]),
               "relative_db":float(power_db(P[k]/P[chosen[0]]))} for i,k in enumerate(chosen)]
    fractions=[float(P[frequency_band_mask(nu,*b)].sum()/P.sum()*100) for b in BANDS]
    centroid=float(np.dot(nu,P)/P.sum())
    row={"instrument":name,"samples":n,"fs":fs,"duration_s":n/fs,"mean_removed":mu,
         "rms":float(np.sqrt(np.mean(x*x))),"rms_active":float(active_rms),
         "active_duration_s":float(counts[active].sum()/fs),"activity_threshold_db":-40,
         "peak_hz":float(nu[chosen[0]]),"centroid_hz":centroid,"energy":energy,
         "energy_fft":ef,"parseval_relative_error":error,"bin_spacing_hz":df,
         "band_0_250_pct":fractions[0],"band_250_1000_pct":fractions[1],
         "band_1000_4000_pct":fractions[2],"band_4000_22050_pct":fractions[3],
         "sha256":hashlib.sha256(source.read_bytes()).hexdigest()}
    pd.DataFrame({"time_s":tr,"rms":rms,"active":active}).to_csv(output/f"rms_{name}.csv",index=False)
    fig,ax=plt.subplots(2,1,figsize=(10,5.4),sharex=True,constrained_layout=True)
    block=441;si=np.arange(0,n,block);te=si/fs
    low=np.minimum.reduceat(x,si);high=np.maximum.reduceat(x,si)
    ax[0].fill_between(te,low,high,color=color,lw=0);ax[0].set(ylabel="Amplitudo digital",title=f"{name}: selubung min–maks 10 ms")
    ax[1].plot(tr,rms,color=color);ax[1].set(xlabel="Waktu (s)",ylabel="RMS",xlim=(0,n/fs),title="RMS per 100 ms; tanpa normalisasi puncak")
    save_figure(fig,f"waktu_{name}.png",figure_dir)
    xx,yy=spectral_envelope(nu,P,.5)
    fig,ax=plt.subplots(2,1,figsize=(10,6),constrained_layout=True)
    for axes,limit in zip(ax,[5000,1000]):
        axes.plot(xx,power_db(yy),color=color,lw=.8);axes.set(xlim=(0,limit),ylim=fft_limits,xlabel="Frekuensi ν (Hz)",ylabel="Daya (dB re 1 amplitudo²/Hz)")
        axes.grid(alpha=.2)
    ax[0].set_title(f"FFT global {name}; selubung maksimum tiap 0,5 Hz")
    for k in chosen[:3]:ax[1].annotate(f"{nu[k]:.1f}",(nu[k],power_db(P[k])),xytext=(5,4),textcoords="offset points",fontsize=8)
    save_figure(fig,f"fft_{name}.png",figure_dir)
    # Ringkasan temporal memakai periodogram segmen aktual, tanpa pengurangan mean per segmen.
    segments=[]
    for start in np.arange(0,n/fs,10):
        part=x[round(start*fs):min(n,round((start+10)*fs))]
        u,p,_=spectrum(part,fs);total=p.sum()
        seg_rms=float(np.sqrt(np.mean(part**2)))
        raw_part=mono[round(start*fs):min(n,round((start+10)*fs))]
        meaningful=np.sqrt(np.mean(raw_part**2))>row["rms_active"]*.01
        segments.append({"instrument":name,"start_s":float(start),"end_s":min(n/fs,start+10),"rms":seg_rms,
           "meaningful":bool(meaningful),"centroid_hz":float(u@p/total) if meaningful else None,
           "low_band_pct":float(p[frequency_band_mask(u,0,250)].sum()/total*100) if meaningful else None,
           "high_band_pct":float(p[frequency_band_mask(u,1000,22050)].sum()/total*100) if meaningful else None})
    stfts=[];validation=[]
    for M in (1024,4096):
        w=signal.windows.hann(M,sym=False);H=M//4
        u,t,S=signal.spectrogram(x,fs,window=w,nperseg=M,noverlap=M-H,nfft=M,
                                 detrend=False,return_onesided=True,scaling="density",mode="psd")
        errors=[]
        for iq in [0,len(t)//2,len(t)-1]:
            frame=x[iq*H:iq*H+M];manual=one_sided_weights(M)*abs(fft.rfft(frame*w))**2/(fs*np.sum(w*w))
            assert np.allclose(S[:,iq],manual,rtol=1e-10,atol=1e-20)
            e1=np.sum((frame*w)**2)/np.sum(w*w);e2=np.sum(S[:,iq])*fs/M
            errors.append(float(abs(e1-e2)/max(e1,1e-30)))
        validation.append({"instrument":name,"M":M,"frames":len(t),"frame_parseval_max_error":max(errors)})
        # Nilai fitur memakai semua bin; gambar tidak dipakai untuk menghitung energi.
        power=S.sum(axis=0);c=u@S/np.maximum(power,1e-30)
        lowband=S[u<250].sum(axis=0)/np.maximum(power,1e-30)*100
        if M==4096:
            pd.DataFrame({"time_s":t,"centroid_hz":c,"low_band_pct":lowband,
                          "power":power*fs/M}).to_csv(output/f"fitur_stft_{name}.csv",index=False)
        if name=="Brass":
            around=(t>=29)&(t<30)
            level=power[around];tt=t[around]
            above=tt[level>=level.max()*.1]
            row[f"brass_onset_10pct_M{M}"]=float(above[0])
        stfts.append((M,u,t,S))
    for full in (True,False):
        fig,ax=plt.subplots(2,1,figsize=(10.8,6.8),sharex=True,sharey=True,constrained_layout=True)
        for axes,(M,u,t,S) in zip(ax,stfts):
            maskf=u<=frequency_max
            if full:
                step=max(1,int(np.ceil(len(t)/1200)));indices=np.arange(0,len(t),step)
                z=np.maximum.reduceat(S[maskf],indices,axis=1)
                tt=np.minimum(t[indices]+(step-1)*(M/4/fs)/2,t[-1]);xr=(0,n/fs)
            else:
                maskt=(t>=ZOOM[0])&(t<=ZOOM[1]);tt=t[maskt];z=S[maskf][:,maskt];xr=ZOOM
            mesh=axes.pcolormesh(tt,u[maskf],power_db(z),shading="auto",cmap="magma",vmin=db_limits[0],vmax=db_limits[1],rasterized=True)
            axes.set(xlim=xr,ylim=(0,frequency_max),ylabel="Frekuensi ν (Hz)",title=f"{name}: Hann M={M}, H={M//4}, Δν={fs/M:.2f} Hz")
        ax[-1].set_xlabel("Pusat jendela τ (s)")
        fig.colorbar(mesh,ax=ax,label="Daya (dB re 1 amplitudo²/Hz)",pad=.02)
        save_figure(fig,f"stft_{'penuh' if full else 'zoom'}_{name}.png",figure_dir)
    del stfts
    return row,peakrows,segments,validation,(te,low,high,tr,rms,xx,yy)

### Perbandingan Lima Instrumen dan Ekspor Hasil

Fungsi ini menerapkan analisis per berkas, menghitung onset awal dan ukuran penyebaran jendela Hann, kemudian menulis CSV, JSON, gambar, dan tabel LaTeX untuk laporan.

In [ ]:
def run_instrument_analysis():
    prepare_analysis_folders();results=[];peaks=[];segments=[];validation=[];views=[]
    for name,color in zip(INSTRUMENT_NAMES,INSTRUMENT_COLORS):
        row,p,s,v,view=analyse_one(name,color)
        results.append(row);peaks+=p;segments+=s;validation+=v;views.append(view)
        print(name,"peak",round(row["peak_hz"],3),"centroid",round(row["centroid_hz"],2),flush=True)
    summary=pd.DataFrame(results);summary.to_csv(INSTRUMENT_OUTPUT/"perbandingan.csv",index=False)
    pd.DataFrame(peaks).to_csv(INSTRUMENT_OUTPUT/"puncak.csv",index=False)
    pd.DataFrame(segments).to_csv(INSTRUMENT_OUTPUT/"segmen.csv",index=False)
    pd.DataFrame(validation).to_csv(INSTRUMENT_OUTPUT/"validasi_stft.csv",index=False)
    fig,ax=plt.subplots(5,2,figsize=(11.5,10),sharex="col",constrained_layout=True)
    for i,(name,col,v) in enumerate(zip(INSTRUMENT_NAMES,INSTRUMENT_COLORS,views)):
        te,lo,hi,tr,rms,xx,yy=v
        ax[i,0].fill_between(te,lo,hi,color=col,lw=0);ax[i,0].set(ylabel=name,ylim=(-.16,.16),xlim=(0,72))
        ax[i,1].plot(tr,rms,color=col,lw=.8);ax[i,1].set(ylim=(0,.065),xlim=(0,72))
        for a in ax[i]:a.grid(alpha=.15)
    ax[0,0].set_title("Amplitudo digital; min–maks 10 ms")
    ax[0,1].set_title("RMS lokal per 100 ms")
    for a in ax[-1]:a.set_xlabel("Waktu (s)")
    save_figure(fig,"01_waveform_rms_perbandingan.png")
    fig,ax=plt.subplots(5,1,figsize=(11,9.2),sharex=True,sharey=True,constrained_layout=True)
    for i,(name,col,v) in enumerate(zip(INSTRUMENT_NAMES,INSTRUMENT_COLORS,views)):
        xx,yy=v[-2:];ax[i].plot(xx,power_db(yy),color=col,lw=.8)
        ax[i].set(ylabel=name,xlim=(0,2500),ylim=(-135,-25));ax[i].grid(alpha=.2)
        ax[i].text(.98,.84,f"puncak terpilih {results[i]['peak_hz']:.2f} Hz",transform=ax[i].transAxes,ha="right",fontsize=9)
    ax[0].set_title("Kerapatan daya global (dB re 1 amplitudo²/Hz); selubung maksimum 0,5 Hz")
    ax[-1].set_xlabel("Frekuensi ν (Hz)");save_figure(fig,"02_fft_perbandingan.png")
    fig,ax=plt.subplots(figsize=(10,4.5),constrained_layout=True)
    values=np.array([[r[f"band_{lo}_{hi}_pct"] for lo,hi in BANDS] for r in results])
    base=np.zeros(5)
    for j,label in enumerate(["0–250 Hz","250–1000 Hz","1000–4000 Hz","4000–22050 Hz"]):
        ax.bar(INSTRUMENT_NAMES,values[:,j],bottom=base,label=label);base+=values[:,j]
    ax.set(ylabel="Fraksi energi (%)",ylim=(0,100),title="Komposisi energi; tinggi total tiap instrumen = 100%")
    ax.legend(loc="upper center",bbox_to_anchor=(.5,1.02),ncol=4,fontsize=8)
    save_figure(fig,"03_pita_energi.png")
    note_df=pd.read_csv(ROOT/"hasil/rekonstruksi/not_timeline.csv")
    onset_metrics=[]
    fig,ax=plt.subplots(5,1,figsize=(10,8),sharex=True,constrained_layout=True)
    for i,name in enumerate(INSTRUMENT_NAMES):
        a,fs=sf.read(ROOT/"hasil/audio/instrumen"/f"{name}.wav");x=a.mean(axis=1)
        part=note_df[note_df.group==name];onset=float(part.start_seconds.min())
        first=part[part.start_seconds==onset];off=float(first.end_seconds.max())
        nexton=float(part.loc[part.start_seconds>onset,"start_seconds"].min())
        start=round(onset*fs);stop=round((onset+.18)*fs);xx=x[start:stop]
        Q=round(.005*fs);si=np.arange(0,len(xx),Q);ct=np.minimum(Q,len(xx)-si)
        env=np.sqrt(np.add.reduceat(xx*xx,si)/ct);tt=(si+ct/2)/fs
        ax[i].plot(tt*1000,env,color=INSTRUMENT_COLORS[i]);ax[i].set(ylabel=name,xlim=(0,180))
        if off-onset<.18:ax[i].axvline((off-onset)*1000,color=".4",ls="--")
        ax[i].grid(alpha=.2)
        onset_metrics.append({"instrument":name,"source_onset_s":onset,"first_note_off_s":off,
                              "next_note_on_s":nexton,"early_rms_peak_ms":float(tt[env.argmax()]*1000),
                              "rms_10_30_ms":float(np.sqrt(np.mean(xx[round(.01*fs):round(.03*fs)]**2))),
                              "rms_60_90_ms":float(np.sqrt(np.mean(xx[round(.06*fs):round(.09*fs)]**2)))})
    ax[0].set_title("RMS 5 ms pada 180 ms pertama sejak not awal; garis putus = note-off")
    ax[-1].set_xlabel("Waktu relatif terhadap note-on pertama (ms)")
    save_figure(fig,"04_attack_awal.png")
    pd.DataFrame(onset_metrics).to_csv(INSTRUMENT_OUTPUT/"attack_awal.csv",index=False)
    bass=note_df[note_df.group=="Bass"].sort_values(["start_tick","pitch_midi"])
    tuba=note_df[note_df.group=="Tuba"].sort_values(["start_tick","pitch_midi"])
    paired=bool(np.array_equal(bass.start_tick.to_numpy(),tuba.start_tick.to_numpy()) and
                np.array_equal(bass.end_tick.to_numpy(),tuba.end_tick.to_numpy()) and
                np.all(tuba.pitch_midi.to_numpy()-bass.pitch_midi.to_numpy()==12))
    end_differences = tuba.end_tick.to_numpy() - bass.end_tick.to_numpy()
    bass_tuba_comparison = {
        "notes_per_instrument": len(bass),
        "same_onsets": bool(np.array_equal(bass.start_tick.to_numpy(), tuba.start_tick.to_numpy())),
        "pitch_difference_semitones": sorted(set((tuba.pitch_midi.to_numpy() - bass.pitch_midi.to_numpy()).tolist())),
        "same_note_offs": bool(np.all(end_differences == 0)),
        "note_off_difference_counts_ticks": {str(int(delta)): int(count) for delta, count in zip(*np.unique(end_differences, return_counts=True))},
    }
    window_rows=[]
    for M in (1024,4096):
        a=M/44100
        window_rows.append({"M":M,"H":M//4,"overlap_pct":75,"duration_ms":a*1000,"hop_ms":M/4/44100*1000,
          "df_hz":44100/M,"enbw_hz":1.5*44100/M,"mainlobe_hz":4*44100/M,
          "sigma_t_ms":a*np.sqrt(1/12-5/(8*np.pi**2))*1000,"sigma_omega":2*np.pi/(np.sqrt(3)*a),
          "product":float(np.sqrt(np.pi**2/9-5/6))})
    pd.DataFrame(window_rows).to_csv(INSTRUMENT_OUTPUT/"jendela.csv",index=False)
    mix,fs=sf.read(ROOT/"hasil/audio/instrumen/All_Instruments.wav")
    sf.write(CLIPS/"All_Instruments_29-33s.wav",mix[round(29*fs):round(33*fs)],fs,subtype="PCM_16")
    payload={"instruments":results,"peaks":peaks,"segments":segments,"validation":validation,
             "attack":onset_metrics,"windows":window_rows,"bass_tuba_same_timing_octave_apart":paired,
             "bass_tuba_comparison":bass_tuba_comparison,
             "zoom_seconds":ZOOM,"db_limits":INSTRUMENT_DB_LIMITS,"versions":{"python":platform.python_version(),"numpy":np.__version__,
                "scipy":scipy.__version__,"soundfile":sf.__version__,"matplotlib":matplotlib.__version__},
             "drums_reference":"hasil/kanal_audio/summary.json; dua kanal sumber independen 160.77 s"}
    (INSTRUMENT_OUTPUT/"summary.json").write_text(json.dumps(payload,indent=2,ensure_ascii=False),encoding="utf-8")
    write_instrument_table("tabel_spektrum.tex",["Instrumen",r"Puncak (Hz)","Sentroid (Hz)",r"$<250$ Hz (\%)",r"$\geq1000$ Hz (\%)"],
              [[r["instrument"],f'{r["peak_hz"]:.2f}',f'{r["centroid_hz"]:.2f}',f'{r["band_0_250_pct"]:.2f}',
                f'{r["band_1000_4000_pct"]+r["band_4000_22050_pct"]:.2f}'] for r in results])
    write_instrument_table("tabel_waktu.tex",["Instrumen","RMS seluruh","RMS aktif","Aktif (s)"],
              [[r["instrument"],f'{r["rms"]:.6f}',f'{r["rms_active"]:.6f}',f'{r["active_duration_s"]:.2f}'] for r in results])
    write_instrument_table("tabel_parseval.tex",["Instrumen",r"$E_t$",r"$E_F$","Galat relatif"],
              [[r["instrument"],f'{r["energy"]:.9f}',f'{r["energy_fft"]:.9f}',f'{r["parseval_relative_error"]:.2e}'] for r in results])
    write_instrument_table("tabel_jendela.tex",[r"$M$",r"$H$","Durasi (ms)","Hop (ms)",r"$\Delta\nu$ (Hz)","ENBW (Hz)"],
              [[r["M"],r["H"],f'{r["duration_ms"]:.3f}',f'{r["hop_ms"]:.3f}',f'{r["df_hz"]:.3f}',f'{r["enbw_hz"]:.3f}'] for r in window_rows])
    valid=pd.read_csv(ROOT/"hasil/rekonstruksi/validasi_audio.csv")
    write_instrument_table("tabel_dataset.tex",["Instrumen","Not","MIDI min--maks","Awal (s)","Akhir not (s)"],
              [[r.instrument,r.notes,f'{r.pitch_min}--{r.pitch_max}',f'{r.first_note_seconds:.3f}',f'{r.last_note_end_seconds:.3f}'] for r in valid.itertuples()])
    print("Analisis lima instrumen selesai.")
    return payload

### Pelaksanaan Analisis Instrumen

RECALCULATE=True menghitung ulang seluruh hasil lima instrumen. Gambar dan cuplikan juga dibuat ulang, sehingga tersedia setelah clone repository.

In [ ]:
RECALCULATE = True
hasil = run_instrument_analysis() if RECALCULATE else json.loads((ROOT/'hasil/instrumen/summary.json').read_text(encoding='utf-8'))
summary = pd.DataFrame(hasil['instruments'])
print('Versi perhitungan:', hasil['versions'])


## 3. Analisis Sinyal dalam Domain Waktu

RMS adalah amplitudo efektif: kuadrat menghilangkan pembatalan tanda positif-negatif, rerata merangkum potongan waktu, dan akar mengembalikan satuan amplitudo. Untuk $Q$ sampel,
$\mathrm{RMS}=\sqrt{Q^{-1}\sum_{k=0}^{Q-1}f_k^2}$.
RMS tinggi tidak otomatis berarti suara dipersepsikan lebih keras; frekuensi, durasi dan pendengaran juga berperan.

Pada gambar pertama, X adalah waktu, kolom kiri menunjukkan amplitudo, dan kanan RMS 100 ms. Skala antarbaris sama. Brass hanya muncul dalam dua kelompok akor. Karena banyak diam, RMS seluruh timeline bisa kecil meskipun RMS saat aktif besar.

Gambar attack memakai X relatif terhadap not pertama dan RMS 5 ms. Garis putus menunjukkan note-off. RMS sependek itu masih dapat berosilasi akibat gelombang, sehingga puncaknya bukan pengukuran murni konstanta attack. Perbedaan ini berlaku untuk preset yang dirender, bukan semua instrumen dengan nama sama.

In [ ]:
display(summary[['instrument','rms','rms_active','active_duration_s']].round(6))
display(Image(filename=str(ROOT/'gambar/instrumen/01_waveform_rms_perbandingan.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/04_attack_awal.png')))
display(pd.read_csv(ROOT/'hasil/instrumen/attack_awal.csv'))


## 4. Analisis Fourier Global Menggunakan FFT

DFT adalah transformasi matematis; FFT adalah algoritma cepat untuk menghitungnya. **Bin** adalah titik frekuensi tempat DFT dievaluasi. Jarak bin $\Delta\nu=f_s/N$ menunjukkan kerapatan kisi, bukan langsung kemampuan memisahkan dua nada.

Mengikuti materi: $F_m=\sum_k f_ke^{-2\pi i mk/N}$, $\nu_m=mf_s/N$, $\omega_m=2\pi\nu_m$, dan $\widehat f(\omega_m)\approx\Delta t F_m/\sqrt{2\pi}$.
Grafik memakai $P_m=c_m|F_m|^2/(f_sN)$, dengan $c_m=2$ pada bin positif interior dan 1 pada DC/Nyquist. Sumbu X frekuensi dan Y daya dalam dB terhadap 1 amplitudo²/Hz. Kurva hanya diringkas dengan maksimum tiap 0,5 Hz; seluruh energi dan puncak dihitung dari bin asli.

Puncak tertinggi menunjukkan kontribusi kuat sepanjang rekaman, bukan otomatis nada dasar. Misalnya brass memainkan akor; harmonik dari beberapa not saling bertumpuk. Diagram pita memperlihatkan persentase energi, bukan identitas instrumen atau suasana musik.

In [ ]:
display(summary[['instrument','peak_hz','centroid_hz','band_0_250_pct','band_250_1000_pct','band_1000_4000_pct','band_4000_22050_pct']].round(3))
display(pd.DataFrame(hasil['peaks']).round(3))
display(Image(filename=str(ROOT/'gambar/instrumen/02_fft_perbandingan.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/03_pita_energi.png')))
display(pd.Series(hasil['bass_tuba_comparison'], name='Perbandingan not Bass dan Tuba'))


## 5. Validasi Energi Menggunakan Identitas Parseval

Perubahan cara melihat sinyal seharusnya tidak mengubah total energinya bila faktor skala benar. Itulah alasan memeriksa $E_t=\Delta t\sum_k|f_k|^2$ terhadap $E_F=(\Delta t/N)\sum_m c_m|F_m|^2$ pada FFT satu sisi. Galat sangat kecil mendukung konsistensi hitungan, bukan keaslian timbre SoundFont terhadap FLEX.

In [ ]:
display(summary[['instrument','energy','energy_fft','parseval_relative_error']])
assert summary.parseval_relative_error.max() < 1e-11
display(pd.DataFrame(hasil['validation']))


## 6. Analisis Waktu–Frekuensi Menggunakan STFT

Jika FFT global membaca seluruh audio sekaligus, STFT menggeser potongan pendek dan menghitung FFT berkali-kali. Sumbu X adalah pusat waktu potongan, Y frekuensi, dan warna daya. Garis mendatar menandakan komponen yang bertahan; garis tegak dapat menandakan kejadian singkat yang menjangkau banyak frekuensi. Interpretasi selalu dikaitkan dengan not yang memang ada.

Materi memakai $V_wf(\tau,\omega)=\frac1{\sqrt{2\pi}}\int f(t)\overline{w(t-\tau)}e^{-i\omega t}dt$ dan $\mathrm{Spec}_wf=|V_wf|^2$. Untuk DFT frame $Z_{q,m}$, $\mathrm{Spec}\approx\Delta t^2|Z|^2/(2\pi)$. Grafik ditampilkan sebagai density satu sisi $P_{q,m}=c_m|Z|^2/(f_s\sum_rw_r^2)$ agar kedua panjang jendela dapat dibandingkan secara konsisten.

Semua panel memakai batas warna −110 sampai −35 dB dengan acuan sama, rentang 0–2000 Hz, serta cuplikan 29–33 s yang sama. Gambar penuh diringkas menggunakan maksimum kelompok frame (sekitar 1200 kolom); gambar zoom tidak diagregasi. Grafik penuh dipakai untuk lokasi aktivitas, bukan mengukur lebar transien.

### Bass

Sebagian besar daya berada di frekuensi rendah. Pergantian not terlihat pada garis rendah yang berubah posisi. Jendela pendek membuat pita rendah tampak lebih tebal; jangan menafsirkan ketebalannya sebagai banyak nada baru.

Cuplikan audio tersedia untuk pemeriksaan pendengaran oleh pembaca. Uraian di atas berasal dari data/not dan pola grafik; tidak mengaku telah melakukan uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/instrumen/stft_penuh_Bass.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/stft_zoom_Bass.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/Bass_29-33s.wav'), embed=True))


### Piano

Aransemen piano memuat beberapa channel dan banyak not. Pola pendek yang berganti menunjukkan pergantian nada/akor. Garis di frekuensi tinggi dapat berupa harmonik atau not lain yang bersamaan, sehingga tidak semua garis mewakili instrumen baru.

Cuplikan audio tersedia untuk pemeriksaan pendengaran oleh pembaca. Uraian di atas berasal dari data/not dan pola grafik; tidak mengaku telah melakukan uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/instrumen/stft_penuh_Piano.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/stft_zoom_Piano.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/Piano_29-33s.wav'), embed=True))


### Guitar

Kelompok ini menggabungkan iringan dan melodi yang penempatannya berasal dari FLP. Saat komponen melodi muncul, pergeseran daya dapat berasal dari register dan not tambahan, bukan perubahan jenis gitar.

Cuplikan audio tersedia untuk pemeriksaan pendengaran oleh pembaca. Uraian di atas berasal dari data/not dan pola grafik; tidak mengaku telah melakukan uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/instrumen/stft_penuh_Guitar.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/stft_zoom_Guitar.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/Guitar_29-33s.wav'), embed=True))


### Brass

Akor kedua mulai pada 29,23074 s dan note-off pada 32,30766 s. Garis mendatar pada interval ini cocok dengan not yang ditahan. Pada jendela 4096, garis sekitar 1,4–1,6 kHz terpisah lebih jelas; pada 1024 lebih menyatu. Titik awal pada gambar dapat bergeser akibat jendela yang dipusatkan, bukan perubahan timing MIDI.

Cuplikan audio tersedia untuk pemeriksaan pendengaran oleh pembaca. Uraian di atas berasal dari data/not dan pola grafik; tidak mengaku telah melakukan uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/instrumen/stft_penuh_Brass.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/stft_zoom_Brass.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/Brass_29-33s.wav'), embed=True))


### Tuba

Seluruh 187 onset not sama dengan bass, dan pitch satu oktaf lebih tinggi. Sebanyak 167 note-off sama; 16 not tuba berakhir 2 tick lebih lambat dan 4 not berakhir 4 tick lebih lambat. Jadi, waktu akhir not tidak seluruhnya identik. Presetnya juga berbeda. Spektrum karenanya tidak harus berupa salinan bass yang hanya diperbesar amplitudonya. Preset GM Tuba di sini bukan rekonstruksi artikulasi Tubas Staccato FLEX yang identik.

Cuplikan audio tersedia untuk pemeriksaan pendengaran oleh pembaca. Uraian di atas berasal dari data/not dan pola grafik; tidak mengaku telah melakukan uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/instrumen/stft_penuh_Tuba.png')))
display(Image(filename=str(ROOT/'gambar/instrumen/stft_zoom_Tuba.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/Tuba_29-33s.wav'), embed=True))


## 7. Pengaruh Panjang Jendela dan Prinsip Ketaktentuan Heisenberg

Potongan pendek membantu mengetahui *kapan* bunyi berubah, tetapi memberi lebih sedikit siklus untuk membedakan frekuensi berdekatan. Potongan panjang melakukan kebalikannya. Hann memperkecil bobot tepi untuk mengurangi kebocoran spektral; konsekuensinya respons spektral memiliki lebar sendiri.

Dipilih M=1024 dan 4096 agar durasinya berbeda tepat empat kali. Bentuk Hann sama, $w_r=\tfrac12-\tfrac12\cos(2\pi r/M)$, hop H=M/4, overlap 75%. Durasi adalah M/fs, hop waktu H/fs, dan jarak bin fs/M. ENBW Hann =1,5 fs/M; lebar lobus utama sekitar4 fs/M. Jadi jarak bin saja bukan resolusi sebenarnya.

Dalam model kontinu, $\sigma_t$ mengukur penyebaran energi jendela terhadap waktu, sedangkan $\sigma_\omega$ penyebarannya terhadap frekuensi sudut. Batasnya $\sigma_t\sigma_\omega\ge1/2$, atau $\sigma_t\sigma_\nu\ge1/(4\pi)$ dalam Hz. Untuk Hann kontinu dengan durasi a: $\sigma_t=a\sqrt{1/12-5/(8\pi^2)}$, $\sigma_\omega=2\pi/(\sqrt3a)$. Produk sekitar0,513 berasal dari **model jendela**, bukan ketelitian deteksi not yang diukur dari audio.

In [ ]:
display(pd.DataFrame(hasil['windows']).round(6))
assert all(r['product'] >= .5 for r in hasil['windows'])
brass = summary.set_index('instrument').loc['Brass']
print('Pusat frame pertama mencapai 10% daya sekitar awal akor kedua:')
print('M1024:', brass['brass_onset_10pct_M1024'], 'M4096:', brass['brass_onset_10pct_M4096'])
print('Nilai ambang ini bukan waktu onset fisik; window panjang bisa tampak lebih awal akibat lokalisasi terpusat.')


## 8. Perubahan Temporal dan Karakteristik Nonstasioner

Sentroid adalah pusat distribusi daya terhadap frekuensi. Nilai yang lebih tinggi berarti pusat daya bergeser ke frekuensi lebih tinggi, bukan berarti amplitudo lebih besar. Dengan bahasa analogi, komposisi demikian bisa disebut relatif lebih terang; ini interpretasi spektral, bukan hasil uji persepsi.

Tabel memakai periodogram tiap segmen 10 s. Segmen hampir senyap tidak diberi interpretasi sentroid. Perubahan RMS, distribusi pita dan not mendukung istilah **nonstasioner secara deskriptif**. Tidak dilakukan pengujian stasioneritas statistik formal. Besarnya perubahan juga berbeda: bass dapat mempertahankan fraksi frekuensi rendah yang relatif stabil sambil notnya tetap berubah.

In [ ]:
segmen = pd.DataFrame(hasil['segments'])
display(segmen[segmen.meaningful].round(4))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/All_Instruments_29-33s.wav'), embed=True))


## 9. Analisis Kanal DRUMS dan VOCALS dalam Format WAV

Channel 11 dan 12 di FLP merujuk ke `DRUMS.mp3` dan `VOCALS.mp3`. Keduanya berupa klip audio, bukan not MIDI untuk disintesis ulang. Kode pada bagian ini mendekode masing-masing sumber, menyimpannya sebagai WAV float32 stereo 44.100 Hz, lalu menjalankan analisis. Semua sampel hasil baca ulang WAV diperiksa identik dengan dekode MP3.

**Konversi MP3 ke WAV tidak meningkatkan kualitas sumber.** MP3 sudah mengalami kompresi lossy; WAV tidak memulihkan detail yang hilang. Karena vektor sampelnya sama, hasil RMS, DFT dan STFT dengan parameter sama juga sama. Manfaat WAV hanya format kerja tanpa kompresi lossy tambahan. `drums_source_comparison` menunjukkan selisih RMS, energi dan puncak terhadap perhitungan langsung MP3, setelah dihitung ulang.

Durasi kedua berkas tetap 160,77 s. Timecode sumber belum disetarakan dengan WAV rekonstruksi 71,23 s, sehingga keduanya tidak dimasukkan ke campuran. Parameter stretching yang terbaca netral belum menggantikan validasi pemetaan offset dan playback terhadap render acuan. Nama VOCALS adalah label berkas, bukan bukti kemurnian suara manusia. Nilai bawaan `RECALCULATE_SOURCE=True` mengulangi konversi dan komputasi, lalu memeriksa hash WAV dan MP3. Nilai `False` dapat dipakai untuk membaca hasil tersimpan ketika seluruh gambar dan audio sudah tersedia.

### Dekode WAV dan Validasi Sampel Sumber

Kanal DRUMS/VOCALS diaudit langsung dari FLP. Hasil dekode MP3 disimpan sebagai WAV float32, lalu setiap sampel dibandingkan kembali. Tidak ada resampling, perubahan amplitudo, atau pemulihan detail yang hilang akibat kompresi MP3.

In [ ]:
SOURCE_WAV = ROOT / "hasil/audio/kanal"
CHANNEL_OUTPUT = ROOT / "hasil/kanal_audio"
CHANNEL_FIGURES = ROOT / "gambar/kanal_audio"
CLIPS = ROOT / "hasil/audio/cuplikan"
CHANNEL_NAMES = ["DRUMS", "VOCALS"]
CHANNEL_COLORS = ["#455c73", "#b24072"]

def convert_sources():
    for directory in (SOURCE_WAV, CHANNEL_OUTPUT, CHANNEL_FIGURES, CLIPS):
        directory.mkdir(parents=True, exist_ok=True)
    folder = ROOT / "data/sumber"
    project = pyflp.parse(next(folder.glob("*.flp")))
    playlist = json.loads((ROOT / "hasil/rekonstruksi/playlist.json").read_text(encoding="utf-8"))
    rows = []
    channel_audit = []
    for name in CHANNEL_NAMES:
        source = folder / f"{name}.mp3"
        channel = next(c for c in project.channels if c.name == name)
        assert channel.sample_path.name == source.name
        audio, fs = sf.read(source, dtype="float32", always_2d=True)
        assert fs == 44100 and audio.shape[1] == 2 and np.isfinite(audio).all()
        assert np.any(audio)
        destination = SOURCE_WAV / f"{name}.wav"
        sf.write(destination, audio, fs, format="WAV", subtype="FLOAT")
        decoded, check_fs = sf.read(destination, dtype="float32", always_2d=True)
        assert check_fs == fs and np.array_equal(audio, decoded)
        assert sf.info(destination).subtype == "FLOAT"
        clip = next(c for c in playlist if c["kind"] == "channel" and c["name"] == name)
        audit = {"name": name, "channel_id": channel.iid,
                 "sample_basename": channel.sample_path.name,
                 "stretch_mode": int(channel.stretching.mode),
                 "stretch_multiplier": float(channel.stretching.multiplier),
                 "stretch_pitch_cents": channel.stretching.pitch,
                 "stretch_time": list(channel.stretching.time),
                 "pitch_shift_cents": channel.pitch_shift,
                 "start_offset_parameter": channel.playback.start_offset,
                 "use_loop_points": channel.playback.use_loop_points,
                 "playlist_position_ticks": clip["position_ticks"],
                 "playlist_length_ticks": clip["length_ticks"],
                 "playlist_offset_start_raw": clip["offset_start_raw"],
                 "playlist_offset_end_raw": clip["offset_end_raw"],
                 "alignment_status": "Berkas penuh; tidak ditempatkan ulang atau dicampur dengan MIDI. Parameter stretching terbaca netral, tetapi pemetaan offset audio dan hasil playback FL Studio belum diuji terhadap render acuan."}
        channel_audit.append(audit)
        row = {"channel": name, "source": str(source.relative_to(ROOT)),
               "wav": str(destination.relative_to(ROOT)), "subtype": "FLOAT",
               "fs": fs, "channels": audio.shape[1], "frames": len(audio),
               "duration_s": len(audio)/fs, "peak": float(np.max(np.abs(audio))),
               "samples_at_or_above_one": int(np.count_nonzero(np.abs(audio) >= 1)),
               "roundtrip_samples_identical": True, "max_abs_conversion_error": 0.0,
               "source_sha256": hashlib.sha256(source.read_bytes()).hexdigest(),
               "wav_sha256": hashlib.sha256(destination.read_bytes()).hexdigest()}
        rows.append(row)
        print(name, f"{len(audio)/fs:.6f} s; WAV float32 identik dengan hasil dekode MP3", flush=True)
    pd.DataFrame(rows).to_csv(CHANNEL_OUTPUT / "validasi_konversi.csv", index=False)
    write_json(CHANNEL_OUTPUT / "validasi_konversi.json", rows)
    write_json(CHANNEL_OUTPUT / "audit_kanal_flp.json", channel_audit)
    return rows, channel_audit

### Analisis dan Ekspor Kedua Kanal

Kedua berkas penuh memakai fungsi analyse_one yang telah didefinisikan di atas. Domain waktu mengikuti awal masing-masing berkas sumber. Hasil DRUMS dari WAV dibandingkan dengan perhitungan langsung terhadap dekode MP3.

In [ ]:
def write_channel_table(results):
    lines = [r"\begin{tabular}{lrrrrr}", r"\toprule",
             r"Kanal & RMS & Puncak (Hz) & Sentroid (Hz) & $<250$ Hz (\%) & $\geq1000$ Hz (\%) \\",
             r"\midrule"]
    for r in results:
        high = r["band_1000_4000_pct"] + r["band_4000_22050_pct"]
        lines.append(f'{r["instrument"]} & {r["rms"]:.5f} & {r["peak_hz"]:.2f} & '
                     f'{r["centroid_hz"]:.2f} & {r["band_0_250_pct"]:.2f} & {high:.2f} ' + r"\\")
    lines.extend([r"\bottomrule", r"\end{tabular}"])
    (CHANNEL_OUTPUT / "tabel_perbandingan.tex").write_text("\n".join(lines) + "\n", encoding="utf-8")

def run_source_analysis():
    converted, channel_audit = convert_sources()
    rows, peaks, segments, validations, views = [], [], [], [], []
    for name, color in zip(CHANNEL_NAMES, CHANNEL_COLORS):
        row, p, s, v, view = analyse_one(
            name, color, source=SOURCE_WAV/f"{name}.wav", output=CHANNEL_OUTPUT,
            figure_dir=CHANNEL_FIGURES, clip_dir=CLIPS, frequency_max=5000,
            db_limits=(-110, -20), fft_limits=(-135, -10))
        rows.append(row); peaks.extend(p); segments.extend(s); validations.extend(v); views.append(view)
        print(name, "puncak", round(row["peak_hz"], 3), "sentroid", round(row["centroid_hz"], 3), flush=True)
    for name, content in [("perbandingan", rows), ("puncak", peaks), ("segmen", segments), ("validasi_stft", validations)]:
        pd.DataFrame(content).to_csv(CHANNEL_OUTPUT/f"{name}.csv", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(11.5, 6.4), sharex=True, constrained_layout=True)
    amplitude_limit = max(float(max(abs(v[1]).max(), abs(v[2]).max())) for v in views)*1.05
    rms_limit = max(float(v[4].max()) for v in views)*1.05
    for i, (name, color, v) in enumerate(zip(CHANNEL_NAMES, CHANNEL_COLORS, views)):
        te, low, high, tr, rms, _, _ = v
        axes[i, 0].fill_between(te, low, high, color=color, lw=0)
        axes[i, 0].set(ylabel=f"{name}\nAmplitudo", ylim=(-amplitude_limit, amplitude_limit))
        axes[i, 1].plot(tr, rms, color=color, lw=.8)
        axes[i, 1].set(ylabel="RMS", ylim=(0, rms_limit))
        for axis in axes[i]: axis.set(xlim=(0, rows[i]["duration_s"])); axis.grid(alpha=.15)
    axes[0, 0].set_title("Amplitudo digital; min–maks 10 ms")
    axes[0, 1].set_title("RMS per 100 ms")
    for axis in axes[-1]: axis.set_xlabel("Waktu dari awal berkas sumber (s)")
    save_figure(fig, "01_waveform_rms_kanal.png", CHANNEL_FIGURES)

    fig, axes = plt.subplots(2, 1, figsize=(11, 6.2), sharex=True, sharey=True, constrained_layout=True)
    for axis, name, color, v in zip(axes, CHANNEL_NAMES, CHANNEL_COLORS, views):
        nu, power = v[-2:]
        axis.plot(nu, power_db(power), color=color, lw=.8)
        axis.set(ylabel=f"{name}\nDaya (dB)", xlim=(0, 5000), ylim=(-135, -10)); axis.grid(alpha=.2)
    axes[0].set_title("Kerapatan daya global; acuan 1 amplitudo²/Hz; maksimum per 0,5 Hz")
    axes[-1].set_xlabel("Frekuensi ν (Hz)")
    save_figure(fig, "02_fft_kanal.png", CHANNEL_FIGURES)
    write_channel_table(rows)
    source_mp3 = ROOT / converted[0]["source"]
    direct_audio, direct_fs = sf.read(source_mp3, always_2d=True)
    direct = direct_audio.mean(axis=1)
    direct -= direct.mean()
    direct_nu, direct_power, _ = spectrum(direct, direct_fs)
    direct_peaks, _ = signal.find_peaks(direct_power, distance=int(np.ceil(10/(direct_fs/len(direct)))))
    direct_peaks = direct_peaks[(direct_nu[direct_peaks]>=20)&(direct_nu[direct_peaks]<=10000)]
    direct_peak = direct_nu[direct_peaks[np.argmax(direct_power[direct_peaks])]]
    # Referensi dihitung langsung dari MP3, tanpa bergantung pada arsip hasil lama.
    payload = {"conversion": converted, "channels": rows, "peaks": peaks, "segments": segments,
               "validation": validations, "flp_audio_channel_audit": channel_audit,
               "stft": {"fs": 44100, "M": [1024, 4096], "H": [256, 1024], "overlap_pct": 75,
                        "window": "Hann periodic", "zoom_source_seconds": [29, 33],
                        "frequency_plot_max_hz": 5000, "db_limits": [-110, -20]},
               "drums_source_reference": str(source_mp3.relative_to(ROOT)),
               "drums_source_comparison": {
                   "rms_abs_difference": abs(rows[0]["rms"]-float(np.sqrt(np.mean(direct**2)))),
                   "energy_abs_difference": abs(rows[0]["energy"]-float(np.sum(direct**2)/direct_fs)),
                   "peak_hz_abs_difference": abs(rows[0]["peak_hz"]-float(direct_peak))},
               "scope": "Dua WAV dari hasil dekode MP3 penuh; bukan render MIDI atau peningkatan kualitas sumber; timecode belum disetarakan dengan lima instrumen."}
    write_json(CHANNEL_OUTPUT/"summary.json", payload)
    return payload

### Pelaksanaan Analisis Kanal Sumber

RECALCULATE_SOURCE=True menjalankan ulang konversi, FFT, STFT, dan ekspor kedua kanal. Semua sampel WAV divalidasi terhadap dekode sumber MP3.

In [ ]:
RECALCULATE_SOURCE = True
required_source = [ROOT/'hasil/kanal_audio/summary.json'] + [ROOT/folder/f'{name}{suffix}' for name in ['DRUMS','VOCALS'] for folder,suffix in [('hasil/audio/kanal','.wav'),('hasil/audio/cuplikan','_29-33s.wav')]]
kanal = run_source_analysis() if RECALCULATE_SOURCE or not all(path.exists() for path in required_source) else json.loads((ROOT/'hasil/kanal_audio/summary.json').read_text(encoding='utf-8'))
for row in kanal['conversion']:
    assert hashlib.sha256((ROOT/row['wav']).read_bytes()).hexdigest() == row['wav_sha256']
    assert hashlib.sha256((ROOT/row['source']).read_bytes()).hexdigest() == row['source_sha256']
    assert row['roundtrip_samples_identical'] and row['max_abs_conversion_error'] == 0
display(pd.DataFrame(kanal['conversion'])[['channel','fs','frames','duration_s','peak','max_abs_conversion_error']])
print('Selisih DRUMS terhadap analisis langsung MP3:', kanal['drums_source_comparison'])
assert all(d == 0 for d in kanal['drums_source_comparison'].values())


### Aktivitas Temporal dan Spektrum Global

Pada grafik waktu, X berasal dari awal berkas penuh. Kolom kiri adalah amplitudo dan kanan RMS 100 ms; skala antarbaris sama. DRUMS aktif sepanjang sebagian besar rekaman. VOCALS mempunyai kelompok kejadian kuat sekitar 31,4–35,2 s, 92,3–100,0 s dan 153,0–155,9 s. Ambang aktivitas relatif yang sama dengan analisis sebelumnya menghasilkan sekitar 14,2 s aktif pada VOCALS; RMS globalnya 0,04895 dan RMS aktifnya 0,16470. Durasi diam menjelaskan perbedaan itu.

Grafik FFT memakai X frekuensi linear dan Y daya per Hz dalam dB dengan acuan sama. DRUMS memiliki sekitar 91,19% energi di bawah 250 Hz. VOCALS terutama pada 250–1000 Hz (68,11%) dan pada 1000 Hz ke atas (31,87%). Puncak VOCALS sekitar 780,08 Hz bukan penetapan otomatis fundamental suara manusia. Perbedaan berasal dari isi audio, bukan perubahan ekstensi ke WAV.

In [ ]:
kanal_summary = pd.DataFrame(kanal['channels'])
display(kanal_summary[['instrument','rms','rms_active','active_duration_s','peak_hz','centroid_hz','band_0_250_pct','parseval_relative_error']].round(6))
display(Image(filename=str(ROOT/'gambar/kanal_audio/01_waveform_rms_kanal.png')))
display(Image(filename=str(ROOT/'gambar/kanal_audio/02_fft_kanal.png')))
display(pd.DataFrame(kanal['validation']))


### Karakteristik Waktu–Frekuensi DRUMS

Daerah terang rendah disertai guratan singkat berulang sekitar 29–31 s. Guratan menjangkau frekuensi lebih tinggi, sesuai transien yang mempunyai kontribusi pada banyak frekuensi. Jendela panjang memperinci garis horizontal tetapi melibatkan rentang waktu lebih lebar.

Panel atas memakai Hann 1024, bawah 4096; hop seperempat panjang jendela. X adalah pusat waktu sumber, Y frekuensi, dan warna daya per Hz. Pasangan DRUMS/VOCALS menggunakan 0–5000 Hz dan −110 hingga −20 dB; jangan membandingkan warna langsung dengan panel instrumen yang memakai batas lain. Pembesaran 29–33 s tidak diagregasi. Audio tersedia untuk pemeriksaan oleh pembaca; tidak ada klaim uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/kanal_audio/stft_penuh_DRUMS.png')))
display(Image(filename=str(ROOT/'gambar/kanal_audio/stft_zoom_DRUMS.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/DRUMS_29-33s.wav'), embed=True))


### Karakteristik Waktu–Frekuensi VOCALS

Sebelum sekitar 31,4 s, panel dominan gelap. Setelahnya muncul jalur terang sekitar 0,8 kHz dan kelipatannya, dengan perubahan posisi sebelum dan sesudah bagian relatif mendatar sekitar 31,8–32,5 s. Pola mendekati kelipatan mendukung struktur harmonik, tetapi tidak menetapkan fundamental atau jenis sumber secara unik.

Panel atas memakai Hann 1024, bawah 4096; hop seperempat panjang jendela. X adalah pusat waktu sumber, Y frekuensi, dan warna daya per Hz. Pasangan DRUMS/VOCALS menggunakan 0–5000 Hz dan −110 hingga −20 dB; jangan membandingkan warna langsung dengan panel instrumen yang memakai batas lain. Pembesaran 29–33 s tidak diagregasi. Audio tersedia untuk pemeriksaan oleh pembaca; tidak ada klaim uji dengar subjektif.

In [ ]:
display(Image(filename=str(ROOT/'gambar/kanal_audio/stft_penuh_VOCALS.png')))
display(Image(filename=str(ROOT/'gambar/kanal_audio/stft_zoom_VOCALS.png')))
display(Audio(filename=str(ROOT/'hasil/audio/cuplikan/VOCALS_29-33s.wav'), embed=True))


## 10. Interpretasi Hasil dan Referensi

Hasil mengukur gabungan karakter preset, not yang dimainkan, velocity, register, dan durasi aktif. Karena partitur tiap instrumen berbeda, penelitian ini bukan eksperimen yang hanya mengubah timbre sambil mempertahankan semua variabel lain. FFT merangkum kontribusi frekuensi; STFT menunjukkan pergantiannya dalam waktu. Rekonstruksi SoundFont berhasil untuk kelima kelompok, tetapi tidak mereproduksi efek/mixing/preset FL Studio.

Rujukan: `../Materi/Kuliah-04-AFW.pdf`, `KuliahAFW-05.pdf`, `KuliahAFW-06.pdf`; [video Rick Ruyz](https://www.youtube.com/watch?v=DdIcPpjQXV4); [PyFLP](https://pyflp.readthedocs.io/); [FluidSynth](https://www.fluidsynth.org/); [GeneralUser GS](https://www.schristiancollins.com/generaluser.php). Sumber dan lisensi SoundFont disertakan, begitu pula hash sumber dan parameter rekonstruksi. Detail konversi Playlist dan audit rekonstruksi dijelaskan pada bagian Pembahasan dalam paper ETS AFW.pdf.